<a href="https://colab.research.google.com/github/girish17work/Data-Analysis/blob/main/Unknown_Dataset_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [53]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=15jD1GGhhMFztGrekLORfiZKHzuNg3K6f'
df = pd.read_csv(DATASET_URL)
df.head()

,Age,Gender,Annual Income,Number of Dependents,Education Level,Health Score,Location,Policy Type,Active Policy days,Previous Claims,Credit Score,Insurance Duration,Customer Feedback,Smoking Status,Exercise Frequency,Property Type,Premium Amount
0,21.0,Female,79975.0,2.0,PhD,48.836351,Urban,Premium,1536,0.0,387.0,8.0,Average,No,Rarely,Apartment,755.0
1,30.0,Male,35244.0,1.0,Bachelor's,36.490112,Urban,Premium,1018,0.0,502.0,3.0,Poor,Yes,Rarely,Condo,1017.0
2,28.0,Female,3868.0,3.0,Bachelor's,20.965763,Urban,Basic,1054,0.0,508.0,4.0,Good,No,Weekly,Apartment,822.0
3,29.0,Male,72466.0,NaN,Bachelor's,NaN,Suburban,Premium,1825,2.0,NaN,8.0,Poor,No,Daily,House,679.0
4,26.0,Female,15845.0,3.0,Master's,24.353805,Suburban,Basic,434,0.0,744.0,5.0,Good,Yes,Weekly,Condo,656.0


In [54]:
print(f'Columns with numerical datatype: {df.select_dtypes(include='number').columns.to_list()}')

Columns with numerical datatype: ['Age', 'Annual Income', 'Number of Dependents', 'Health Score', 'Active Policy days', 'Previous Claims', 'Credit Score', 'Insurance Duration', 'Premium Amount']


In [55]:
print(f'Most common policy type among customers aged 35 or below, living in Suburban locations and having a credit score of 600 or above: {df[(df['Age']<=35) & (df['Location']=='Suburban') & (df['Credit Score']>600)]['Policy Type'].mode()[0]}')

Most common policy type among customers aged 35 or below, living in Suburban locations and having a credit score of 600 or above: Comprehensive


In [56]:
print(f'Unique categories in the Exercise frequency: {df['Exercise Frequency'].unique().tolist()}')

Unique categories in the Exercise frequency: ['Rarely', 'Weekly', 'Daily', 'Monthly']


In [57]:
X = df.drop(columns=['Premium Amount'])
y = df['Premium Amount']

print(X.select_dtypes(include='number').corr())

                           Age  Annual Income  Number of Dependents  \
Age                   1.000000       0.024984             -0.000111   
Annual Income         0.024984       1.000000             -0.006069   
Number of Dependents -0.000111      -0.006069              1.000000   
Health Score         -0.009154       0.014707              0.004596   
Active Policy days    0.003363      -0.010889             -0.001713   
Previous Claims      -0.001410       0.032629             -0.003458   
Credit Score         -0.014900      -0.201620             -0.005398   
Insurance Duration   -0.009305       0.003672              0.004855   

                      Health Score  Active Policy days  Previous Claims  \
Age                      -0.009154            0.003363        -0.001410   
Annual Income             0.014707           -0.010889         0.032629   
Number of Dependents      0.004596           -0.001713        -0.003458   
Health Score              1.000000           -0.008358      

In [58]:
print(f'Columns with missing values: {df.columns[df.isna().any()].tolist()}')

Columns with missing values: ['Age', 'Annual Income', 'Number of Dependents', 'Health Score', 'Credit Score', 'Customer Feedback']


In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [60]:
q1 = y_test.quantile(0.25)
q3 = y_test.quantile(0.75)
iqr = q3 - q1
lb = q1 - 1.5*iqr
ub = q3 + 1.5*iqr
print(f'Outlier: {((y_test<lb) | (y_test>ub)).sum()}')

Outlier: 117


In [61]:
fill_values = {
    'Age': X_train['Age'].mean(),
    'Annual Income': X_train['Annual Income'].median(),
    'Number of Dependents': X_train['Number of Dependents'].mode()[0],
    'Health Score': X_train['Health Score'].mean(),
    'Credit Score': X_train['Credit Score'].median(),
    'Customer Feedback': X_train['Customer Feedback'].mode()[0]
}

X_train = X_train.fillna(fill_values)
X_test = X_test.fillna(fill_values)

In [62]:
print(f'Mean of Age: {round(X_train['Age'].mean(), 1)}')

Mean of Age: 41.3


In [63]:
print(fill_values)

{'Age': np.float64(41.27006904020577), 'Annual Income': 23991.0, 'Number of Dependents': np.float64(3.0), 'Health Score': np.float64(25.519598071693363), 'Credit Score': 602.0, 'Customer Feedback': 'Good'}


In [64]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

ordinal_cols = ['Education Level', 'Exercise Frequency']
nominal_cols = ['Gender', 'Smoking Status', 'Location', 'Policy Type', 'Customer Feedback', 'Property Type']

preprocessor = Pipeline([
    ('ct', ColumnTransformer([
        ('oe', OrdinalEncoder(), ordinal_cols),
        ('ohe', OneHotEncoder(sparse_output=False, drop='first'), nominal_cols)
    ], remainder='passthrough').set_output(transform='pandas')),
    ('scale', MinMaxScaler())
]).set_output(transform='pandas')

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [65]:
print(f'Sum of first 50 rows in the Test Feature Matrix: {X_test[:50].to_numpy().sum()}')

Sum of first 50 rows in the Test Feature Matrix: 405.3393527191712


In [66]:
DATASET_URL = 'https://drive.google.com/uc?export=download&id=16EU5JD4jXWdRsnESjOCdS484WAUV5f-E'
df = pd.read_csv(DATASET_URL)
df.head()

,Education Level,Exercise Frequency,Gender_Male,Smoking Status_Yes,Location_Suburban,Location_Urban,Policy Type_Comprehensive,Policy Type_Premium,Customer Feedback_Good,Customer Feedback_Poor,...,Property Type_House,Age,Health Score,Annual Income,Number of Dependents,Credit Score,Active Policy days,Previous Claims,Insurance Duration,Premium Amount
0,-1.346013,-1.357284,-1.005013,-0.983388,-0.704589,1.399493,1.407347,-0.712151,-0.778733,-0.673612,...,-0.695866,0.570618,0.468016,-0.389475,-1.519638,-2.230924,-0.822925,-0.747169,-1.517374,2538.0
1,1.352985,1.321346,0.995012,1.016893,-0.704589,-0.714545,-0.710557,1.404197,-0.778733,-0.673612,...,1.437057,-1.572652,-0.149257,3.691485,-1.519638,-2.230924,-1.276273,0.324425,-1.134851,112.0
2,-0.446347,-1.357284,-1.005013,-0.983388,-0.704589,1.399493,-0.710557,1.404197,-0.778733,-0.673612,...,1.437057,0.866242,-1.362263,-0.473263,1.379224,0.875385,-0.058985,1.396019,1.542812,2431.0
3,-1.346013,0.428469,-1.005013,-0.983388,-0.704589,1.399493,1.407347,-0.712151,-0.778733,-0.673612,...,-0.695866,-1.498747,-1.105478,1.115244,-1.519638,0.824601,1.742831,-0.747169,0.777766,2546.0
4,1.352985,-0.464408,0.995012,-0.983388,-0.704589,1.399493,-0.710557,1.404197,-0.778733,-0.673612,...,-0.695866,-1.203123,0.969650,-0.911842,-0.794922,1.247804,-1.554069,1.396019,1.542812,807.0


In [67]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Premium Amount'])
y = df['Premium Amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [68]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

ridge = Ridge(alpha=5, solver='saga', tol=1e-4, random_state=42)
ridge.fit(X_train, y_train)

y_pred = ridge.predict(X_test)
print(f'Mean Absolute Error: {round(mean_absolute_error(y_test, y_pred), 2)}')

Mean Absolute Error: 666.01


In [70]:
print(f'Highest absolute value of coefficient of ridge trained model: {round(abs(ridge.coef_).max(), 2)}')

Highest absolute value of coefficient of ridge trained model: 36.55


In [71]:
print(f'Index of feature with the lowest absolute coeffcient: {np.argmin(np.abs(ridge.coef_))}')

Index of feature with the lowest absolute coeffcient: 16


In [72]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV

estimator = SGDRegressor(random_state=42)
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    'tol': [1e-3, 1e-2]
}

grid_search_cv = GridSearchCV(estimator=estimator, param_grid=param_grid, cv=5, scoring='neg_mean_squared_error', refit=True)
grid_search_cv.fit(X_train, y_train)

best_estimator = grid_search_cv.best_estimator_

In [73]:
print(f'Best Alpha: {grid_search_cv.best_params_['alpha']}')

Best Alpha: 0.1


In [74]:
from sklearn.metrics import root_mean_squared_error

y_pred = best_estimator.predict(X_test)
print(f'Root mean squared error on the test dataset: {round(root_mean_squared_error(y_test, y_pred), 2)}')

Root mean squared error on the test dataset: 872.39


In [75]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso

pipeline = Pipeline([('pca', PCA()), ('lasso', Lasso())])

param_grid = {
    'pca__n_components': [0.85, 0.95],
    'lasso__alpha': [10, 1, 0.1, 0.01]
}

grid_search_cv = GridSearchCV(pipeline, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search_cv.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('pca', PCA()), ('lasso', Lasso())]),
             n_jobs=-1,
             param_grid={'lasso__alpha': [10, 1, 0.1, 0.01],
                         'pca__n_components': [0.85, 0.95]},
             scoring='neg_mean_absolute_error')

In [76]:
from sklearn.metrics import mean_absolute_error

y_pred = grid_search_cv.predict(X_test)
print(f'Mean Absolute Error on the test dataset: {round(mean_absolute_error(y_test, y_pred), 2)}')

Mean Absolute Error on the test dataset: 665.48


In [90]:
print(f'Number of principal components retained by the best pipeline: {grid_search_cv.best_estimator_.named_steps['pca'].n_components_}')

Number of principal components retained by the best pipeline: 15


In [95]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error

pipeline = Pipeline([('poly', PolynomialFeatures(interaction_only=False, degree=2)), ('lasso', Lasso(alpha=2, warm_start=True, random_state=0))])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print(f'Root Mean Squared Error: {round(root_mean_squared_error(y_test, y_pred), 2)}')

Root Mean Squared Error: 872.52


In [111]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import SGDRegressor

pipeline = Pipeline([('kbest', SelectKBest(f_regression, k=4)), ('sgd', SGDRegressor(random_state=42))])
pipeline.fit(X_train, y_train)

kbest = pipeline.named_steps['kbest']
top = np.argsort(kbest.scores_)[::-1][:4]
print(f'Selected Features: {X_train.columns[top].to_list()}')

Selected Features: ['Previous Claims', 'Location_Urban', 'Annual Income', 'Active Policy days']


In [114]:
sgd = pipeline.named_steps['sgd']
print(f'Actual number of iterations completed during model training: {sgd.n_iter_}')

Actual number of iterations completed during model training: 16
